#  Incremental SCD Type 2 MERGE for customers

### What we're changing
**Before:** `write.mode("overwrite")` — rebuilds the entire Silver customers table every run
**After:** `MERGE INTO ... USING ...` — only applies the delta, keeps history

### The SCD Type 2 pattern
For every customer, we keep multiple rows representing their state over time:
```
customer_id  | service_tier | valid_from   | valid_to     | is_current
CUST_000001  | Bronze       | 2024-01-15   | 2025-06-15   | false
CUST_000001  | Gold         | 2025-06-15   | NULL         | true
```

Two rows for one customer. To get current state, filter `WHERE is_current = true`.
To get historical state, filter `WHERE valid_from <= '2025-03-15' AND (valid_to IS NULL OR valid_to > '2025-03-15')`.

### Pipeline run logging
Every run records: start time, duration, rows in/inserted/updated, status.
This is the audit log that proves to a stakeholder that yesterday's batch ran successfully.




In [0]:
#Config + imports
CATALOG = "cx"
SCHEMA_BRONZE = "cx_bronze"
SCHEMA_SILVER = "cx_silver"
DELTA_PATH = "/Volumes/cx/default/cx_project_raw_delta"  

from pyspark.sql.functions import (
    col, current_timestamp, current_date, lit, sha2, concat_ws,
    when, initcap, trim, to_date, coalesce
)
from pyspark.sql.types import IntegerType, DoubleType
from datetime import datetime

# Pipeline run tracking
run_start = datetime.now()
run_id = run_start.strftime("%Y%m%d_%H%M%S")
print(f"Pipeline run started: {run_id}")


Pipeline run started: 20260604_190935


## Step 1 — Create the SCD2 Silver table (one-time setup)

If `customers_scd2` doesn't exist, we initialize it from the current Silver customers table.
This is a "first-run" bootstrap. On subsequent runs, this cell is a no-op.

In [0]:
#Bootstrap SCD2 table from existing Silver
scd2_table = f"{CATALOG}.{SCHEMA_SILVER}.customers_scd2"

# Check if SCD2 table exists; if not, bootstrap from existing Silver
table_exists = spark.catalog.tableExists(scd2_table)

if not table_exists:
    print(f"Bootstrapping {scd2_table} from existing Silver customers...")
    silver_customers = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.customers")

    # Add SCD2 columns
    bootstrap = (
        silver_customers
        .withColumn("valid_from", to_date(col("contract_start_date")))
        .withColumn("valid_to", lit(None).cast("date"))
        .withColumn("is_current", lit(True))
        .withColumn(
            "row_hash",
            sha2(concat_ws("||",
                col("customer_type"),
                col("service_tier"),
                col("annual_recurring_revenue_usd").cast("string"),
                col("primary_contact_email"),
            ), 256)
        )
    )

    bootstrap.write.mode("overwrite").format("delta").saveAsTable(scd2_table)
    print(f"  Initialized with {bootstrap.count():,} rows")
else:
    print(f"{scd2_table} already exists — skipping bootstrap")

Bootstrapping cx.cx_silver.customers_scd2 from existing Silver customers...
  Initialized with 50,000 rows


## Step 2 — Ingest the delta into Bronze

The delta CSV gets landed in Bronze first (same as the initial ingestion pattern).

In [0]:
#Bronze data Ingestion
bronze_delta_table = f"{CATALOG}.{SCHEMA_BRONZE}.customers_incoming"

bronze_delta = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv(f"{DELTA_PATH}/customers_delta.csv")
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_file", lit("customers_delta.csv"))
    .withColumn("pipeline_run_id", lit(run_id))
)

bronze_delta.write.mode("overwrite").format("delta").saveAsTable(bronze_delta_table)
delta_count = bronze_delta.count()
print(f"Bronze delta ingested: {delta_count:,} rows")

Bronze delta ingested: 550 rows


## Step 3 — Clean the delta (same DQ rules as before)

Apply identical cleansing to the delta as we did originally. Same rules → same trusted format.
Compute `row_hash` over the tracked attributes to detect changes.

In [0]:
#Clean delta + compute change-detection hash
delta_clean = (
    spark.table(bronze_delta_table)
    .withColumn("customer_id", trim(col("customer_id")))
    .withColumn("customer_name", trim(col("customer_name")))
    .withColumn("customer_type", trim(col("customer_type")))
    .withColumn("region", trim(col("region")))
    .withColumn("country", trim(col("country")))
    .withColumn("service_tier", initcap(trim(col("service_tier"))))
    .withColumn("contract_start_date", to_date(col("contract_start_date")))
    .withColumn("annual_recurring_revenue_usd", col("annual_recurring_revenue_usd").cast(DoubleType()))
    .withColumn("employee_count", col("employee_count").cast(IntegerType()))
    .withColumn("primary_contact_email", trim(col("primary_contact_email")))
    .dropDuplicates(["customer_id"])
    # Hash the SCD-tracked attributes (the ones whose changes we care about)
    .withColumn(
        "row_hash",
        sha2(concat_ws("||",
            col("customer_type"),
            col("service_tier"),
            col("annual_recurring_revenue_usd").cast("string"),
            col("primary_contact_email"),
        ), 256)
    )
)

delta_clean.createOrReplaceTempView("delta_clean")
print(f"Delta cleaned and hashed: {delta_clean.count():,} rows")

Delta cleaned and hashed: 550 rows


Step 4 — The SCD Type 2 MERGE (the core pattern)

For each row in the delta:
- **If the customer doesn't exist in SCD2:** INSERT a new row (valid_from = today, is_current = true)
- **If the customer exists AND the row_hash matches:** do nothing (no real change)
- **If the customer exists AND the row_hash differs:** the customer's attributes changed
  - Close the existing current record (set valid_to = today, is_current = false)
  - Insert a new record with the new attributes (valid_from = today, is_current = true)

We do this in **two MERGE operations** because Spark SQL MERGE doesn't support
"close-then-insert" in a single statement. This is the standard SCD2 pattern.

In [0]:
#Close out changed records
# Find rows where customer exists in SCD2 (is_current = true) but row_hash has changed
spark.sql(f"""
    MERGE INTO {scd2_table} AS target
    USING (
        SELECT d.customer_id, d.row_hash
        FROM delta_clean d
        INNER JOIN {scd2_table} t
          ON d.customer_id = t.customer_id
          AND t.is_current = true
        WHERE d.row_hash != t.row_hash
    ) AS source
    ON target.customer_id = source.customer_id AND target.is_current = true
    WHEN MATCHED THEN UPDATE SET
        target.valid_to = current_date(),
        target.is_current = false
""")
print("Step 4a complete: closed-out records with changed row_hash")

Step 4a complete: closed-out records with changed row_hash


In [0]:
#Insert new versions and brand-new customers
spark.sql(f"""
    MERGE INTO {scd2_table} AS target
    USING (
        SELECT
            d.customer_id, d.customer_name, d.customer_type, d.region, d.country,
            d.service_tier, d.contract_start_date, d.annual_recurring_revenue_usd,
            d.employee_count, d.primary_contact_email, d.row_hash
        FROM delta_clean d
        LEFT JOIN {scd2_table} t
          ON d.customer_id = t.customer_id AND t.is_current = true
        WHERE
            -- Brand-new customer (no existing row) OR existing customer with changed hash
            t.customer_id IS NULL OR d.row_hash != t.row_hash
    ) AS source
    ON target.customer_id = source.customer_id
       AND target.is_current = true
       AND target.row_hash = source.row_hash
    WHEN NOT MATCHED THEN INSERT (
        customer_id, customer_name, customer_type, region, country,
        service_tier, contract_start_date, annual_recurring_revenue_usd,
        employee_count, primary_contact_email,
        valid_from, valid_to, is_current, row_hash
    ) VALUES (
        source.customer_id, source.customer_name, source.customer_type, source.region, source.country,
        source.service_tier, source.contract_start_date, source.annual_recurring_revenue_usd,
        source.employee_count, source.primary_contact_email,
        current_date(), NULL, true, source.row_hash
    )
""")
print("Step 4b complete: new records inserted")

Step 4b complete: new records inserted


## Step 5 — Log the run

Every production pipeline logs run metadata. This is the audit trail.

In [0]:
#Pipeline Run Log
from pyspark.sql.functions import expr

run_end = datetime.now()
duration_sec = (run_end - run_start).total_seconds()

# Get the new row counts
total_scd_rows = spark.table(scd2_table).count()
current_rows = spark.table(scd2_table).filter("is_current = true").count()
historical_rows = total_scd_rows - current_rows

run_log_table = f"{CATALOG}.{SCHEMA_SILVER}.pipeline_run_log"

run_log_data = [(
    run_id,
    "customers_scd2_merge",
    run_start,
    run_end,
    float(duration_sec),
    delta_count,
    current_rows,
    historical_rows,
    "SUCCESS",
)]

run_log_df = spark.createDataFrame(
    run_log_data,
    schema="run_id STRING, pipeline_name STRING, start_time TIMESTAMP, end_time TIMESTAMP, duration_seconds DOUBLE, rows_in_delta INT, current_rows INT, historical_rows INT, status STRING"
)

run_log_df.write.mode("append").format("delta").saveAsTable(run_log_table)
print(f"\nRun logged: {run_id}")
print(f"  Duration: {duration_sec:.1f}s")
print(f"  Delta rows in: {delta_count:,}")
print(f"  Current rows in SCD2: {current_rows:,}")
print(f"  Historical rows in SCD2: {historical_rows:,}")


Run logged: 20260604_190935
  Duration: 1165.3s
  Delta rows in: 550
  Current rows in SCD2: 50,500
  Historical rows in SCD2: 41


## Step 6 — Verify the SCD2 logic worked

The point of SCD2 is that history is preserved. Pick a customer that was updated
(tier changed) and verify both the old and new records exist.

In [0]:
%sql
--Find customers with multiple versions
 SELECT customer_id, COUNT(*) AS versions
 FROM cx.cx_silver.customers_scd2
 GROUP BY customer_id
 HAVING COUNT(*) > 1
 ORDER BY versions DESC
 LIMIT 10;

customer_id,versions
CUST_043352,2
CUST_028457,2
CUST_006463,2
CUST_008411,2
CUST_020540,2
CUST_026956,2
CUST_018961,2
CUST_018097,2
CUST_008142,2
CUST_034317,2


In [0]:
%sql
-- Drill into one customer's history
--Pick a customer_id from the result above, then:
 SELECT customer_id, service_tier, annual_recurring_revenue_usd,
        valid_from, valid_to, is_current, row_hash
 FROM cx.cx_silver.customers_scd2
 WHERE customer_id = 'CUST_028457'   -- replace with one from above
 ORDER BY valid_from;

customer_id,service_tier,annual_recurring_revenue_usd,valid_from,valid_to,is_current,row_hash
CUST_028457,Silver,52230.78,2019-12-07,2026-06-04,false,af021b5c01d5158944ea2347a180ac3add2238b8ad3a4d23c947f56459fd59dc
CUST_028457,Silver,52230.78,2026-06-04,null,true,516f0d793d780c06d54e61e5841e452ef543e2b25b4b9c258719ca58bc9f8b4d


In [0]:
%sql
--Current state vs historical query examples
 -- Current state (what most queries use)
 SELECT COUNT(*) AS current_customers
 FROM cx.cx_silver.customers_scd2
 WHERE is_current = true;




current_customers
50500


In [0]:
%sql
 -- Time travel: how many customers were Gold-tier on a specific date?
 SELECT COUNT(*) AS gold_customers_on_date
 FROM cx.cx_silver.customers_scd2
 WHERE service_tier = 'Gold'
   AND valid_from <= '2026-01-01'
   AND (valid_to IS NULL OR valid_to > '2026-01-01');

gold_customers_on_date
7603


## Step 7 — Idempotency proof

A production pipeline must be **idempotent**: running it twice produces the same result, not duplicates.

If you re-run this notebook with the same delta input, you should see ZERO new rows
because every customer_id + row_hash combination already exists in SCD2.

This is what proves the MERGE is correct.

In [0]:
#dempotency check (optional, run a second time)

# If you re-ran cells 6-11 above with the same delta:
# Row count of SCD2 should be UNCHANGED from the first run.

current_total = spark.table(scd2_table).count()
print(f"Current SCD2 total rows: {current_total:,}")
print(f"Run this cell after a second pipeline run to verify idempotency.")
print(f"  If second run was idempotent: row count is unchanged.")
print(f"  If it duplicated: row count grew → there's a bug in the MERGE keys.")

Current SCD2 total rows: 50,541
Run this cell after a second pipeline run to verify idempotency.
  If second run was idempotent: row count is unchanged.
  If it duplicated: row count grew → there's a bug in the MERGE keys.


In [0]:
%sql
DESCRIBE HISTORY cx.cx_silver.customers_scd2;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
4,2026-06-04T19:28:59.000Z,71399775570304,aayushamrute02@gmail.com,MERGE,"Map(predicate -> [""(((customer_id#16433 = customer_id#15962) AND is_current#16449) AND (row_hash#16450 = row_hash#15982))""], clusterBy -> [], matchedPredicates -> [], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(1682456744458168),2bc36272-0899-4539-a1a7-226c5c7b20e1,0604-191652-mbr0po2o-v2n,3,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 0, numTargetBytesAdded -> 0, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 2000, materializeSourceTimeMs -> 11, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 0, numTargetRowsUpdated -> 0, numOutputRows -> 0, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 0, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1957)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
3,2026-06-04T19:28:56.000Z,71399775570304,aayushamrute02@gmail.com,MERGE,"Map(predicate -> [""((customer_id#16160 = customer_id#15962) AND is_current#16176)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [])",null,List(1682456744458168),056a0efe-353e-4c54-868a-eba01decafee,0604-191652-mbr0po2o-v2n,2,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 0, numTargetBytesAdded -> 0, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 1920, materializeSourceTimeMs -> 734, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1139, numTargetRowsUpdated -> 0, numOutputRows -> 0, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 0, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 0)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
2,2026-06-04T19:20:49.000Z,71399775570304,aayushamrute02@gmail.com,MERGE,"Map(predicate -> [""(((customer_id#13385 = customer_id#12252) AND is_current#13401) AND (row_hash#13402 = row_hash#12272))""], clusterBy -> [], matchedPredicates -> [], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(1682456744458168),56839abf-4ab6-4b5e-9588-13dafd7b64a4,0604-191652-mbr0po2o-v2n,1,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 32109, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 1997, materializeSourceTimeMs -> 11, numTargetRowsInserted -> 541, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 0, numTargetRowsUpdated -> 0, numOutputRows -> 541, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 541, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1937)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
1,2026-06-04T19:20:27.000Z,71399775570304,aayushamrute02@gmail.com,MERGE,"Map(predicate -> [""((customer_id#12450 = customer_id#12252) AND is_current#12466)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySource

In [0]:
%sql
SELECT customer_id, customer_type, service_tier,
       annual_recurring_revenue_usd, primary_contact_email,
       valid_from, is_current
FROM cx.cx_silver.customers_scd2
WHERE customer_id = 'CUST_028457'
ORDER BY valid_from;

customer_id,customer_type,service_tier,annual_recurring_revenue_usd,primary_contact_email,valid_from,is_current
CUST_028457,Hospital,Silver,52230.78,contact_28457@example.com,2019-12-07,false
CUST_028457,Hospital,Silver,52230.78,newcontact_028457@example.com,2026-06-04,true
